# 📨 CHRUTH — Messages par appel d'offres (clé API ou Ollama)

Choisis un AO **intéressant** (CHAUD/TIÈDE), l'IA rédige un **email structuré + un script d'appel** à l'acheteur, prêts à copier/coller et modifier.

**Moteur automatique :** si une **clé cloud** est dans `.env` → cloud (rapide, structuré) ; sinon **Ollama** local s'il tourne ; sinon **brouillon déterministe**.

**Précision :** remplis `config_chruth/fiche_chruth.md` avec les vrais faits CHRUTH — l'IA n'utilisera que ça.

**Étapes :** 1) Setup — 2) repère le n° de l'AO — 3) mets-le dans `CHOIX` — 4) Générer. Le brouillon est aussi écrit dans `output/messages_ao/AO_<id>.md` (éditable).

## Setup

In [ ]:
import sys, pathlib, importlib
sys.path.insert(0, str(pathlib.Path.cwd()))
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
import llm_client, prospect_messages as pm, ao_messages as am
from ao_db import connect
from ao_config import AO_DB_PATH
importlib.reload(llm_client); importlib.reload(pm); importlib.reload(am)

FICHE = pm.fiche_chruth()
_moteur = llm_client.moteur_auto()
print('Moteur IA :', _moteur or 'aucun -> brouillon deterministe')
print('Fiche CHRUTH :', 'chargee' if FICHE else 'VIDE (remplis config_chruth/fiche_chruth.md)')

with connect(AO_DB_PATH) as _c:
    AOS = [dict(r) for r in _c.execute(
        "SELECT * FROM ao_records WHERE priorite IN ('CHAUD','TIEDE') "
        "ORDER BY CAST(score_chruth AS INTEGER) DESC").fetchall()]
print(len(AOS), 'AO CHAUD/TIEDE disponibles')

## 1. Liste des AO — repère le numéro voulu

In [ ]:
for i, a in enumerate(AOS):
    print(f"[{i:>3}] {str(a.get('priorite','')):6} | "
          f"{str(a.get('objet',''))[:70]:70} | {a.get('acheteur','')}")

## 2. Choisis un AO

In [ ]:
CHOIX = 0   # numero de l'AO dans la liste ci-dessus

## ▶️ Générer — message structuré prêt à copier/coller

In [ ]:
ao = AOS[CHOIX]
msg = am.generer_message_ao(ao, fiche=FICHE)
chemin = am.ecrire_brouillon_md(ao, msg)
print('AO       :', ao.get('objet', ''))
print('Acheteur :', ao.get('acheteur', ''), '| Ville :', ao.get('ville', ''),
      '| Date limite :', ao.get('date_limite', ''))
print('Source   :', msg['source'], '(ia = redige par le modele ; defaut = brouillon type)')
print('Fichier editable :', chemin)
print('\n' + '=' * 72 + '\nEMAIL\n' + '=' * 72)
print(msg['email'])
print('\n' + '=' * 72 + "\nSCRIPT D'APPEL\n" + '=' * 72)
print(msg['script'])

## Astuces
- Change `CHOIX` et relance *Générer* pour un autre AO.
- **Plus de précision** : édite `config_chruth/fiche_chruth.md` (faits réels CHRUTH).
- **Style** : édite `prompt_ao()` dans `ao_messages.py`.
- Le brouillon éditable est dans `output/messages_ao/`, dans le cockpit `output/AO_CHRUTH.xlsm`, et injecté dans le **mail d'alerte** AO.